# 🗂️ AI Task Prioritizer (Student Edition)
### Memprediksi dan mengurutkan prioritas tugas kuliah dengan Random Forest + SHAP + ipywidgets

**Disusun oleh:** Muhammad Wisnu Jaya Wardana
**Program:** Mini Project, Pelatihan Elektrifikasi 2026

---

## 📌 Ringkasan Eksekutif

| | |
|---|---|
| **Masalah** | Mahasiswa sering salah memilih tugas mana yang dikerjakan duluan. |
| **Solusi** | Model yang memprediksi prioritas (Low / Medium / High / Critical), memberi **skor 0–100** untuk mengurutkan to-do list, dan **menjelaskan alasannya**. |
| **Data** | 200 tugas **sintetis** dengan aturan pembuat label yang diketahui, sehingga kebenaran penjelasan model bisa **diverifikasi**. |
| **Model** | Pipeline (imputasi + encoding) + Random Forest 100 pohon, ditambah **lapisan keputusan berbasis biaya**. |

**Temuan utama**
1. Random Forest mencapai akurasi **±73%** (cross-validation 5×10), jauh di atas baseline (38–66%). **99,5%** kesalahannya hanya meleset 1 tingkat (*quadratic weighted kappa* ±0,86).
2. **Lapisan biaya** memangkas kasus "tugas darurat dianggap biasa" dari **±19% menjadi ±9%** tanpa menurunkan akurasi.
3. **SHAP terverifikasi**: model menemukan dua faktor utama aturan asli (tekanan waktu dan dampak), sekaligus terdeteksi **±12% jalan pintas** lewat fitur yang tidak ada di aturan.
4. **Temuan jujur:** Logistic Regression (±85%) lebih cocok untuk data ini. Random Forest tetap dipakai sesuai brief, dan temuan ini dilaporkan sebagai rekomendasi.
5. **Audit label** menemukan **4 dari 4** label yang sengaja dirusak, di 10 kandidat teratas dari 200 tugas.

Setiap angka yang kodenya tidak ada di alur utama dibuktikan di **Lampiran A–F** di akhir notebook ini.

## 🧭 Daftar Isi
1. Setup · 2. Formulasi Masalah & Data · 3. Pemeriksaan & Pembersihan · 4. EDA · 5. Encoding & Split · 6. Model · 7. Evaluasi · 8. Penjelasan Model (SHAP) · 9. Dari Prediksi ke Keputusan · 10. Aplikasi Interaktif & Demo 3 Skenario · 11. Model Card & Kesimpulan · Lampiran A–F · Versi Library

> ▶️ **Cara menjalankan:** `Runtime → Run all`. Notebook ini **mandiri**: tidak perlu mount Drive atau upload file. Total waktu ±4–6 menit (sel terlama: cross-validation di 7.2 dan Lampiran A & D).

## 1. Setup

In [ ]:
import subprocess, sys, warnings
try:
    import shap
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'shap'], check=True)
    import shap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import (train_test_split, RepeatedStratifiedKFold, StratifiedKFold,
                                     cross_validate, cross_val_predict, cross_val_score, learning_curve)
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, cohen_kappa_score, make_scorer, recall_score)
from sklearn.inspection import permutation_importance
from scipy.stats import spearmanr

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
ORDER = ['Low', 'Medium', 'High', 'Critical']
WARNA = {'Low': '#4C9F70', 'Medium': '#E0B100', 'High': '#E67E22', 'Critical': '#C0392B'}
print('Siap. shap', shap.__version__)

## 2. Formulasi Masalah & Data

### 2.1 Formulasi masalah
| | |
|---|---|
| **Pertanyaan** | Diberi ciri-ciri sebuah tugas, seberapa prioritas tugas itu? |
| **Jenis masalah** | Klasifikasi **multikelas** dengan kelas **berurutan** (Low < Medium < High < Critical) |
| **Input (fitur)** | `task_category`, `deadline_days`, `urgency`, `impact_score`, `estimated_hours`, `complexity`, `dependency_count` |
| **Output (target)** | `priority_level`, ditambah skor 0–100 untuk mengurutkan to-do list |
| **Kesalahan terburuk** | Tugas darurat dianggap biasa (*under-prioritize*), jadi **recall Critical** diawasi khusus |
| **Ukuran keberhasilan** | Jauh di atas baseline sederhana, kesalahan jarang meleset lebih dari 1 tingkat, dan alasan model bisa dijelaskan |

### 2.2 Kenapa data sintetis?
Data tugas mahasiswa sungguhan tidak tersedia. Daripada memakai data acak tanpa pola, dataset ini dibuat oleh **generator dengan aturan yang diketahui**. Keuntungannya:
- Kualitas pembersihan data bisa dicek terhadap nilai aslinya.
- Penjelasan model (SHAP) bisa dicek apakah **benar-benar menemukan** aturan pembuat label. Hal ini mustahil dilakukan pada data nyata.

**Aturan pembuat label** (disembunyikan dari model, hanya dipakai untuk validasi):

$$\text{skor} = 0{,}40\cdot\text{pressure}_n + 0{,}30\cdot\text{impact}_n + 0{,}15\cdot\text{urgency}_n + 0{,}15\cdot\text{deps}_n, \qquad \text{pressure} = \frac{\text{estimated\_hours}}{\text{deadline\_days}\times 3}$$

Subskrip $n$ = dinormalkan ke 0–1. Skor lalu dipotong menjadi 4 kelas. Beberapa label digeser satu tingkat (*noise* manusia), dan data mentah sengaja dikotori: sel kosong, salah ketik kategori, nilai mustahil, dan baris kembar.

**Generator, bagian 1:** konfigurasi dan pembuat ciri-ciri tiap tugas, realistis per kategori.

In [ ]:
import numpy as np
import pandas as pd

SEED = 42
N_TASKS = 200
HOURS_PER_DAY = 3.0  # ASUMSI: jam efektif per hari yang bisa dipakai untuk SATU tugas

rng = np.random.default_rng(SEED)

COURSES = ["Alin", "Statprob", "PBP", "SDA", "Matdis 2", "Basdat", "OS", "Jarkom"]

CATEGORIES = ["Tugas Kuliah", "Praktikum", "Kuis/Ujian", "Organisasi", "Pribadi"]
CATEGORY_PROBS = [0.30, 0.20, 0.20, 0.15, 0.15]


# ---------------------------------------------------------------------------
# 1. Membuat ciri-ciri (feature) tiap tugas, realistis per kategori
# ---------------------------------------------------------------------------
def make_task(i):
    cat = rng.choice(CATEGORIES, p=CATEGORY_PROBS)
    course = rng.choice(COURSES)

    if cat == "Tugas Kuliah":
        name = f"{rng.choice(['Tugas', 'PR', 'Tugas Kelompok', 'Paper'])} {course}"
        deadline = int(rng.integers(1, 22))
        hours = rng.uniform(1, 10)
        impact = int(rng.choice([2, 3, 4], p=[0.3, 0.45, 0.25]))
        deps = int(rng.choice([0, 1, 2, 3], p=[0.55, 0.2, 0.15, 0.1]))
    elif cat == "Praktikum":
        name = f"{rng.choice(['Laporan Praktikum', 'Lab', 'Tutorial'])} {course}"
        deadline = int(rng.integers(1, 10))
        hours = rng.uniform(1.5, 6)
        impact = int(rng.choice([2, 3, 4], p=[0.35, 0.45, 0.2]))
        deps = int(rng.choice([0, 1], p=[0.8, 0.2]))
    elif cat == "Kuis/Ujian":
        kind = rng.choice(["Kuis", "UTS", "UAS"], p=[0.6, 0.2, 0.2])
        name = f"Belajar {kind} {course}"
        deadline = int(rng.integers(1, 22))
        if kind == "Kuis":
            hours, impact = rng.uniform(1, 4), int(rng.choice([2, 3], p=[0.4, 0.6]))
        else:
            hours, impact = rng.uniform(4, 14), 5
        deps = 0
    elif cat == "Organisasi":
        name = rng.choice(["Proposal acara", "Rapat divisi", "Desain poster", "Notulensi rapat",
                           "LPJ kegiatan", "Survei lokasi", "Rekap absensi"])
        deadline = int(rng.integers(1, 15))
        hours = rng.uniform(0.5, 5)
        impact = int(rng.choice([1, 2, 3], p=[0.3, 0.45, 0.25]))
        deps = int(rng.choice([0, 1, 2, 3, 4, 5], p=[0.1, 0.2, 0.25, 0.2, 0.15, 0.1]))
    else:  # Pribadi
        name = rng.choice(["Cuci baju", "Update CV", "Belajar bahasa Inggris", "Bayar kos",
                           "Beres-beres kamar", "Olahraga", "Belanja bulanan"])
        deadline = int(rng.integers(1, 31))
        hours = rng.uniform(0.5, 3)
        impact = int(rng.choice([1, 2], p=[0.6, 0.4]))
        deps = 0

    hours = round(hours * 2) / 2  # bulatkan ke kelipatan 0.5 jam
    hours = max(hours, 0.5)

    # complexity: tugas yang lama cenderung lebih sulit (dengan sedikit acak)
    if hours >= 7:
        complexity = rng.choice(["Medium", "High"], p=[0.3, 0.7])
    elif hours >= 3:
        complexity = rng.choice(["Low", "Medium", "High"], p=[0.2, 0.6, 0.2])
    else:
        complexity = rng.choice(["Low", "Medium"], p=[0.75, 0.25])

    # urgency: PERASAAN "mendesak" manusia, terutama dipicu dekatnya deadline (+ noise)
    if deadline <= 1:
        urgency = 5
    elif deadline <= 3:
        urgency = 4
    elif deadline <= 7:
        urgency = 3
    elif deadline <= 14:
        urgency = 2
    else:
        urgency = 1
    if rng.random() < 0.30:  # 30% orang menilai beda 1 tingkat
        urgency += int(rng.choice([-1, 1]))
    urgency = int(np.clip(urgency, 1, 5))

    return {
        "task_id": f"TASK-{i:03d}",
        "task_name": name,
        "task_category": cat,
        "deadline_days": deadline,
        "urgency": urgency,
        "impact_score": impact,
        "estimated_hours": hours,
        "complexity": complexity,
        "dependency_count": deps,
    }

**Generator, bagian 2:** aturan tersembunyi pembuat label (kunci jawaban). Model tidak pernah melihat fungsi ini.

In [ ]:
# ---------------------------------------------------------------------------
# 2. ATURAN TERSEMBUNYI (kunci jawaban). Model TIDAK pernah melihat fungsi ini.
# ---------------------------------------------------------------------------
def true_priority_score(row):
    """
    Skor 0..1. Semakin besar, semakin prioritas.

    pressure = jam_dibutuhkan / jam_tersedia
             = estimated_hours / (deadline_days * HOURS_PER_DAY)
      > 1 artinya waktunya sudah tidak cukup kalau baru mulai sekarang.
    """
    pressure = row["estimated_hours"] / (row["deadline_days"] * HOURS_PER_DAY)
    pressure_n = min(pressure, 1.5) / 1.5          # 0..1
    impact_n = (row["impact_score"] - 1) / 4        # 0..1
    urgency_n = (row["urgency"] - 1) / 4            # 0..1
    deps_n = min(row["dependency_count"], 4) / 4    # 0..1

    return 0.40 * pressure_n + 0.30 * impact_n + 0.15 * urgency_n + 0.15 * deps_n


LABELS = ["Low", "Medium", "High", "Critical"]
THRESHOLDS = [0.21, 0.32, 0.44]  # batas skor antar kelas (dikalibrasi ~ Low 35%, Med 30%, High 22%, Crit 13%)


def score_to_label(score):
    idx = int(np.searchsorted(THRESHOLDS, score, side="right"))
    return LABELS[idx]

**Generator, bagian 3:** merakit dataset, menambah *noise* label, lalu sengaja mengotori data mentah.

In [ ]:
# ---------------------------------------------------------------------------
# 3. Rakit dataset + noise label (manusia tidak 100% konsisten)
# ---------------------------------------------------------------------------
def build():
    df = pd.DataFrame([make_task(i + 1) for i in range(N_TASKS)])
    score = df.apply(true_priority_score, axis=1)
    labels = score.apply(score_to_label)

    # 8% label bergeser 1 tingkat: meniru ketidakkonsistenan manusia saat melabeli
    flip = rng.random(N_TASKS) < 0.08
    for i in np.where(flip)[0]:
        k = LABELS.index(labels[i]) + int(rng.choice([-1, 1]))
        labels[i] = LABELS[int(np.clip(k, 0, 3))]
    df["priority_level"] = labels

    # ---- 4. Sengaja "mengotori" data mentah untuk latihan Bab 2 ----
    raw = df.copy()
    raw.loc[rng.choice(N_TASKS, 5, replace=False), "estimated_hours"] = np.nan
    raw.loc[rng.choice(N_TASKS, 3, replace=False), "complexity"] = np.nan
    # salah ketik: kategori ASLI-nya tetap, hanya penulisannya yang berantakan
    # (huruf kecil/besar tidak konsisten, spasi berlebih di depan/belakang)
    typo_style = [str.lower, lambda s: s + " ", lambda s: " " + s, str.upper]
    typo_rows = rng.choice(N_TASKS, 4, replace=False)
    for r, style in zip(typo_rows, typo_style):
        raw.loc[r, "task_category"] = style(raw.loc[r, "task_category"])
    outlier_rows = rng.choice(N_TASKS, 2, replace=False)
    raw.loc[outlier_rows[0], "estimated_hours"] = 80.0  # salah ketik "8.0"
    raw.loc[outlier_rows[1], "urgency"] = 7             # di luar skala 1-5
    dup = raw.loc[rng.choice(N_TASKS, 4, replace=False)]
    raw = pd.concat([raw, dup], ignore_index=True)
    raw = raw.sample(frac=1, random_state=SEED).reset_index(drop=True)  # acak urutan

    return df, raw

truth_df, raw = build()            # truth_df = versi bersih (kunci jawaban), raw = data mentah yang dikotori
raw.to_csv('tugas_kuliah_dataset_raw.csv', index=False)
print('Data mentah:', raw.shape)
raw.head()

## 3. Pemeriksaan & Pembersihan Data

### 3.1 Kontrak data: semua aturan kualitas dalam satu fungsi
Daripada memeriksa satu per satu, semua aturan ditulis sebagai **kontrak**. Fungsi yang sama dipakai untuk data mentah dan data bersih.

In [ ]:
KATEGORI_RESMI = {'tugas kuliah': 'Tugas Kuliah', 'praktikum': 'Praktikum', 'kuis/ujian': 'Kuis/Ujian',
                  'organisasi': 'Organisasi', 'pribadi': 'Pribadi'}

def urgency_harapan(hari):
    # aturan wajar: makin dekat deadline, makin mendesak
    return 5 if hari <= 1 else 4 if hari <= 3 else 3 if hari <= 7 else 2 if hari <= 14 else 1

def cek_kontrak(data):
    aturan = {
        'task_id unik'          : data['task_id'].duplicated(keep=False),
        'baris kembar'          : data.duplicated(keep=False),
        'kategori sah'          : ~data['task_category'].isin(set(KATEGORI_RESMI.values())),
        'complexity sah'        : ~data['complexity'].isin({'Low', 'Medium', 'High'}),
        'urgency 1–5'           : ~data['urgency'].between(1, 5),
        'impact 1–5'            : ~data['impact_score'].between(1, 5),
        'jam 0,5–20'            : ~data['estimated_hours'].between(0.5, 20),
        'deadline ≥ 1'          : ~(data['deadline_days'] >= 1),
        'urgency konsisten (±1)': (data['urgency'] - data['deadline_days'].apply(urgency_harapan)).abs() >= 2,
    }
    return pd.DataFrame([{'aturan': k, 'pelanggaran': int(m.sum()),
                          'contoh': ', '.join(data.loc[m, 'task_id'].astype(str).unique()[:4])}
                         for k, m in aturan.items() if m.sum() > 0], columns=['aturan', 'pelanggaran', 'contoh'])

cek_kontrak(raw)

Ditemukan 5 jenis masalah: **sel kosong**, **baris kembar**, **penulisan kategori tidak seragam**, **urgency = 7** (di luar skala 1–5), dan **80 jam** (kemungkinan salah ketik 8,0).

### 3.2 Pembersihan
Urutannya penting: rapikan teks → buang duplikat → nilai mustahil jadi kosong → isi sel kosong.

**Strategi imputasi** (mengisi dari tugas yang mirip, **tanpa** memakai label agar tidak bocor):
| Kolom | Cara isi | Alasan |
|---|---|---|
| `estimated_hours` | median per (kategori, impact) | UAS/UTS (impact 5) jauh lebih lama dari kuis |
| `urgency` | median per kelompok deadline | urgensi dipicu dekatnya deadline |
| `complexity` | modus per kelompok jam kerja | tugas yang lama cenderung lebih sulit |

Pilihan ini **diuji dengan simulasi 200×** (menyembunyikan nilai yang diketahui lalu menebaknya): median kategori+impact memberi MAE ±1,5 jam, setara KNN-imputer dan jauh lebih baik dari median global (±2,3 jam). *(Bukti: Lampiran A. Alasan mengisi per kelompok, bukan median global: Lampiran B.)*

In [ ]:
df = raw.copy()
df['task_category'] = df['task_category'].str.strip().str.lower().map(KATEGORI_RESMI)
df = df.drop_duplicates()

for kolom, (lo, hi) in {'urgency': (1, 5), 'impact_score': (1, 5),
                        'estimated_hours': (0.5, 20), 'deadline_days': (1, 365)}.items():
    df[kolom] = df[kolom].where(df[kolom].isna() | df[kolom].between(lo, hi))   # nilai mustahil -> NaN

df = df.set_index('task_id')
kosong = df.isna()
print('Sel kosong sebelum diisi:', kosong.sum()[kosong.sum() > 0].to_dict())

df['estimated_hours'] = (df['estimated_hours']
                         .fillna(df.groupby(['task_category', 'impact_score'])['estimated_hours'].transform('median'))
                         .fillna(df['estimated_hours'].median()))
kel_deadline = pd.cut(df['deadline_days'], bins=[0, 3, 7, 14, 400])
df['urgency'] = df['urgency'].fillna(df.groupby(kel_deadline, observed=True)['urgency'].transform('median')).round().astype(int)
kel_jam = pd.cut(df['estimated_hours'], bins=[0, 3, 7, 100], right=False)
df['complexity'] = df['complexity'].fillna(df.groupby(kel_jam, observed=True)['complexity'].transform(lambda s: s.mode().iloc[0]))

df = df.sort_values('task_id').reset_index()
df.to_csv('tugas_kuliah_dataset_clean.csv', index=False)
df = pd.read_csv('tugas_kuliah_dataset_clean.csv')   # baca ulang: semua langkah berikutnya memakai FILE yang dikumpulkan
print('Data bersih:', df.shape, '| sel kosong:', int(df.isna().sum().sum()))
print('Kontrak data:', 'LOLOS semua aturan ✅' if cek_kontrak(df).empty else cek_kontrak(df))

### 3.3 Verifikasi terhadap kunci jawaban
Karena data sintetis, setiap sel yang diisi bisa dibandingkan dengan nilai aslinya.

In [ ]:
t = truth_df.set_index('task_id'); d = df.set_index('task_id')
cek = pd.DataFrame([{'task_id': i, 'kolom': k, 'diisi': d.loc[i, k], 'asli': t.loc[i, k]}
                    for k in ['estimated_hours', 'urgency', 'complexity'] for i in kosong.index[kosong[k]]])
cek['tepat'] = cek['diisi'] == cek['asli']
print('Kolom lain identik dengan data asli:',
      all((d[c] == t[c].reindex(d.index)).all() for c in ['task_category', 'deadline_days', 'impact_score', 'dependency_count', 'priority_level']))
cek

**Catatan jujur:** imputasi adalah tebakan terdidik (rata-rata meleset ±2 jam pada 6 sel jam). Uji sensitivitas menunjukkan pilihan metode imputasi praktis **tidak mengubah** akurasi model (akurasi CV 0,728 vs 0,730), karena hanya 10 dari 2.000 sel yang terpengaruh. *(Bukti: Lampiran A.)*

## 4. Exploratory Data Analysis

### 4.1 Statistik deskriptif

In [ ]:
print('Distribusi Target Priority Level (%):')
print((df['priority_level'].value_counts(normalize=True) * 100).reindex(ORDER).round(1).to_string())
print('\nDistribusi Kategori Tugas Mahasiswa:')
print(df['task_category'].value_counts().to_string())
df.describe().round(2)

Kelas tidak seimbang (Low 37,5% vs Critical 13,5%), dan semua rentang angka masuk akal: tidak ada nilai mustahil yang tersisa setelah pembersihan.

### 4.2 Distribusi tiap fitur
- **Bar chart:** jumlah tugas per kelas prioritas.
- **Histogram:** sebaran nilai tiap fitur angka. Fitur bernilai bulat kecil (skala 1–5, jumlah dependensi) digambar per nilai (*discrete*).
- **Boxplot:** apakah nilai fitur **bergeser** antar kelas? Kotak = 50% data tengah, garis di dalamnya = median.

In [ ]:
NUM = ['deadline_days', 'urgency', 'impact_score', 'estimated_hours', 'dependency_count']
fig, axes = plt.subplots(1, 6, figsize=(19, 3.3))
jml = df['priority_level'].value_counts().reindex(ORDER)
axes[0].bar(ORDER, jml.values, color=[WARNA[k] for k in ORDER]); axes[0].bar_label(axes[0].containers[0])
axes[0].set_ylim(0, jml.max() * 1.15); axes[0].set_title('Bar chart: kelas prioritas'); axes[0].tick_params(axis='x', labelsize=8)
for ax, k in zip(axes[1:], NUM):
    kontinu = k in ('deadline_days', 'estimated_hours')
    sns.histplot(df[k], ax=ax, color='#4C72B0', **({'bins': 15} if kontinu else {'discrete': True}))
    ax.set_title(f'Histogram: {k}')
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(19, 3.6))
for ax, k in zip(axes, NUM):
    sns.boxplot(data=df, x='priority_level', y=k, order=ORDER, hue='priority_level', hue_order=ORDER,
                palette=WARNA, legend=False, ax=ax)
    ax.set_xlabel(''); ax.set_title(f'Boxplot: {k}'); ax.tick_params(axis='x', labelsize=8)
plt.tight_layout(); plt.show()

**Bacaan distribusi**
- **Histogram:** `deadline_days` dan `estimated_hours` **menceng ke kanan**: kebanyakan tugas dekat deadline dan singkat, sedikit yang sangat lama. `dependency_count` hampir selalu 0.
- **Boxplot:** makin tinggi prioritas, `deadline_days` makin **kecil**, sedangkan `urgency`, `impact_score`, dan `estimated_hours` makin **besar**. Ini pola yang diharapkan.
- `dependency_count` **tidak** naik rapi: median-nya 0 di hampir semua kelas, dan yang tertinggi justru Medium (banyak tugas Organisasi). Sinyalnya lemah, dan ini nanti terkonfirmasi di SHAP.

### 4.3 Kategori & kompleksitas vs prioritas
`complexity` berupa teks berurutan (Low < Medium < High), jadi tidak bisa digambar sebagai boxplot. Hubungannya dilihat lewat tabel silang. Tabel kedua memakai **proporsi per baris** supaya kelompok yang jumlahnya berbeda bisa dibandingkan.

In [ ]:
from IPython.display import display
display(pd.crosstab(df['task_category'], df['priority_level'])[ORDER])
pd.crosstab(df['complexity'], df['priority_level'], normalize='index').reindex(['Low', 'Medium', 'High'])[ORDER].round(2)

### 4.4 Hubungan antar fitur
`pressure` = jam yang dibutuhkan ÷ jam yang tersedia (asumsi 3 jam per hari). Kolom bantu EDA dibuat di salinan `eda` supaya `df` tetap bersih.

In [ ]:
eda = df.copy()
eda['pressure'] = eda['estimated_hours'] / (eda['deadline_days'] * 3)
eda['y'] = eda['priority_level'].map({k: i for i, k in enumerate(ORDER)})

fig, ax = plt.subplots(figsize=(7, 4.2))
sns.scatterplot(data=eda, x='deadline_days', y='estimated_hours', hue='priority_level', hue_order=ORDER,
                palette=WARNA, s=45, ax=ax)
x = np.linspace(0.5, 30, 100)
ax.plot(x, 3 * x, 'k--', lw=1, label='pressure = 1'); ax.plot(x, 1.5 * x, 'k:', lw=1, label='pressure = 0,5')
ax.set_ylim(0, 15); ax.legend(fontsize=8); ax.set_title('Deadline vs jam kerja'); plt.show()

num = NUM + ['pressure']
korelasi_target = eda[num].corrwith(eda['y'], method='spearman')
print('Korelasi Spearman dengan prioritas:')
print(korelasi_target.sort_values().round(2).to_string())
print('\nKorelasi deadline_days vs urgency:', round(eda[['deadline_days', 'urgency']].corr(method='spearman').iloc[0, 1], 2))

**Temuan EDA**
- **Kelas tidak seimbang:** Critical hanya 27 tugas (13,5%). Di data uji 20% hanya ada ±5 Critical, jadi metrik per kelas sangat berisik.
- **Interaksi:** deadline dan jam kerja masing-masing lemah, tapi **digabung** jadi `pressure` (jam butuh ÷ jam tersedia), korelasinya tertinggi (+0,78). Batas antar kelas **miring**, bukan tegak lurus sumbu.
- **Redundansi:** `urgency` hampir kembar dengan `deadline_days` (−0,90), jadi kredit penjelasan akan terbagi.
- **Jalan pintas potensial:** kategori `Pribadi` hampir selalu Low (29 dari 33, tabel 4.3).
- **Kompleksitas:** makin sulit tugasnya, makin sering prioritasnya tinggi (porsi Critical naik dari 10% di Low ke 23% di High). Tapi `complexity` **tidak ada** di aturan pembuat label; ia ikut berkorelasi karena tugas yang sulit cenderung butuh jam lebih banyak.
- Kekuatan tiap korelasi diuji dengan *bootstrap*, dan hubungan non-linear dengan *mutual information* *(Lampiran C)*.

## 5. Encoding & Train/Test Split

### 5.1 Kode template brief, dan masalahnya
Langkah pertama: menjalankan kode Sesi 2 **persis seperti di brief**, sebagai pembanding.

In [ ]:
# Kode persis dari brief (Sesi 2)
X_tpl = pd.get_dummies(df.drop(columns=['task_id', 'task_name', 'priority_level']), drop_first=True)
y_tpl = df['priority_level']
Xtr_t, Xte_t, ytr_t, yte_t = train_test_split(X_tpl, y_tpl, test_size=0.2, random_state=42, stratify=y_tpl)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(Xtr_t, ytr_t)
y_pred = rf_model.predict(Xte_t)
print(classification_report(yte_t, y_pred))
print('Jumlah kolom hasil get_dummies (200 tugas):', X_tpl.shape[1])

Pada split ini template mencapai akurasi 0,82, sedikit di atas Pipeline di 7.1 (0,775). Dengan cross-validation 5×10, keduanya **setara** (±0,72 vs ±0,73, Lampiran D.5). Jadi Pipeline dipilih bukan karena lebih akurat, tetapi karena **aman dipakai** untuk tugas baru:

In [ ]:
tugas_baru = df.drop(columns=['task_id', 'task_name', 'priority_level']).iloc[[0]]
X_baru = pd.get_dummies(tugas_baru, drop_first=True)
print('Jumlah kolom untuk 1 tugas baru:', X_baru.shape[1], list(X_baru.columns))
try:
    rf_model.predict(X_baru)
except ValueError as e:
    print('❌ Error:', str(e).splitlines()[0])

Pada satu baris, setiap kolom kategori hanya punya **satu** nilai, sehingga `get_dummies(drop_first=True)` membuang semua kolom kategorinya. Jumlah kolom tidak lagi cocok dengan model. Masalah lainnya:
- `complexity` diperlakukan sebagai kategori **tanpa urutan**, padahal Low < Medium < High.
- Tidak ada tempat untuk imputasi kalau pengguna mengosongkan kolom.

### 5.2 Solusi: Pipeline
- **Nominal** (`task_category`) → one-hot; **ordinal** (`complexity`) → Low=0 < Medium=1 < High=2; angka tetap.
- `task_id` dan `task_name` dibuang (pengenal, bukan fitur).
- Semua langkah yang belajar dari data dibungkus dalam **Pipeline**, sehingga hanya melihat data latih. Ini mencegah *data leakage* (bukti bahayanya: Lampiran D), dan input satu baris di aplikasi selalu menghasilkan 11 kolom yang sama.
- Split 80/20 dengan `stratify=y`, sama seperti template. Tanpa stratify, jumlah Critical di data uji bisa 0–12 tergantung acakan *(Lampiran D)*.

In [ ]:
FITUR_ANGKA, FITUR_ORDINAL, FITUR_NOMINAL = (['deadline_days', 'urgency', 'impact_score', 'estimated_hours', 'dependency_count'],
                                             ['complexity'], ['task_category'])
FITUR = FITUR_ANGKA + FITUR_ORDINAL + FITUR_NOMINAL

def buat_preprocess(angka=FITUR_ANGKA):
    return ColumnTransformer(
        [('num', SimpleImputer(strategy='median'), angka),
         ('ord', OrdinalEncoder(categories=[['Low', 'Medium', 'High']]), FITUR_ORDINAL),
         ('nom', OneHotEncoder(handle_unknown='ignore', sparse_output=False), FITUR_NOMINAL)],
        verbose_feature_names_out=False).set_output(transform='pandas')

def buat_pipeline(model, angka=FITUR_ANGKA):
    return Pipeline([('pre', buat_preprocess(angka)), ('model', model)])

X, y = df[FITUR].copy(), df['priority_level'].copy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print('Train:', X_train.shape, '| Test:', X_test.shape)
print('Kolom setelah preprocess, 1 tugas baru:', buat_preprocess().fit(X_train).transform(X.iloc[[0]]).shape[1])
pd.DataFrame({'train': y_train.value_counts().reindex(ORDER), 'test': y_test.value_counts().reindex(ORDER)}).T

## 6. Model: Random Forest

**Cara kerja singkat.** Satu pohon keputusan memilih pertanyaan ya/tidak yang paling menurunkan *Gini impurity* $G = 1 - \sum_k p_k^2$ (peluang dua tugas acak berbeda kelas). Pohon tunggal cenderung **menghafal** data latih. Random Forest melatih banyak pohon pada **bootstrap** berbeda (±63,2% data unik per pohon), dengan **subset fitur acak** di setiap node, lalu merata-ratakan peluangnya. Kesalahan tiap pohon saling menutupi. Pemilihan 100 pohon dicek dengan skor OOB *(Lampiran E)*.

In [ ]:
rf = buat_pipeline(RandomForestClassifier(n_estimators=100, random_state=42)).fit(X_train, y_train)
pohon = buat_pipeline(DecisionTreeClassifier(random_state=42)).fit(X_train, y_train)
for nama, m in [('Satu pohon', pohon), ('Random Forest', rf)]:
    print(f'{nama:14} akurasi latih = {m.score(X_train, y_train):.3f} | uji = {m.score(X_test, y_test):.3f}')

Akurasi latih 100% pada keduanya adalah tanda menghafal. Karena itu penilaian **hanya** memakai data yang tidak dilihat model.

## 7. Evaluasi

### 7.1 Data uji (40 tugas)

In [ ]:
pred = rf.predict(X_test)
cm = confusion_matrix(y_test, pred, labels=ORDER)
ConfusionMatrixDisplay(cm, display_labels=ORDER).plot(cmap='Blues', colorbar=False)
plt.title('Confusion matrix (baris = sebenarnya, kolom = tebakan)'); plt.show()
print(classification_report(y_test, pred, labels=ORDER, digits=2))
a = accuracy_score(y_test, pred)
print(f'Galat standar akurasi dengan 40 data uji: ±{np.sqrt(a * (1 - a) / len(y_test)):.3f}')

Kesalahan di **bawah** diagonal = *under-prioritize* (tugas darurat dianggap biasa), yang paling berbahaya. Untuk aplikasi ini, **recall Critical** adalah metrik terpenting.

### 7.2 Cross-validation 5×10 + baseline + model penantang
Data uji 40 tugas terlalu kecil (galat standarnya dihitung di atas), jadi semua model dinilai ulang dengan **50 split** yang sama. ⏱️ ±1 menit.

In [ ]:
class AturanPressure(BaseEstimator, ClassifierMixin):
    """Baseline tanpa belajar: ambang pada rasio jam butuh / jam tersedia."""
    def fit(self, X, y): self.classes_ = np.array(sorted(ORDER)); return self
    def predict(self, X):
        p = X['estimated_hours'] / (X['deadline_days'] * 3.0)
        return np.array(ORDER)[np.searchsorted([0.12, 0.25, 0.50], p, side='right')]

kandidat = {
    'Dummy (selalu Low)' : DummyClassifier(strategy='most_frequent'),
    'Aturan pressure'    : AturanPressure(),
    'Satu pohon'         : buat_pipeline(DecisionTreeClassifier(random_state=42)),
    'Random Forest'      : buat_pipeline(RandomForestClassifier(n_estimators=100, random_state=42)),
    'Logistic Regression': Pipeline([('pre', buat_preprocess()), ('skala', StandardScaler()),
                                     ('model', LogisticRegression(max_iter=3000, C=10))]),
}
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=0)
skoring = {'akurasi': 'accuracy', 'f1_macro': 'f1_macro',
           'recall_critical': make_scorer(recall_score, labels=['Critical'], average='macro')}
hasil_cv = {nama: cross_validate(m, X, y, cv=cv, scoring=skoring) for nama, m in kandidat.items()}

tabel = pd.DataFrame({nama: {k: f"{r['test_'+k].mean():.3f} ± {r['test_'+k].std():.3f}" for k in skoring}
                      for nama, r in hasil_cv.items()}).T
selisih = hasil_cv['Logistic Regression']['test_akurasi'] - hasil_cv['Random Forest']['test_akurasi']
print(f'LogReg − RF (berpasangan, 50 split): {selisih.mean():+.3f}; LogReg menang di {(selisih > 0).mean():.0%} split')
tabel

### 7.3 Seberapa parah kesalahannya?
Prioritas itu **berurutan**. Salah menebak Critical sebagai High tidak separah menebaknya sebagai Low. *Quadratic weighted kappa* menghukum kesalahan sebanding dengan **kuadrat jaraknya** (1 = sempurna, 0 = setara acak).

In [ ]:
IDX = {k: i for i, k in enumerate(ORDER)}
pred_cv = cross_val_predict(kandidat['Random Forest'], X, y, cv=StratifiedKFold(5, shuffle=True, random_state=0))
jarak = pd.Series(pred_cv).map(IDX).values - y.map(IDX).values
print(f'Meleset ≤ 1 tingkat : {(np.abs(jarak) <= 1).mean():.1%}')
print(f'Under-prioritize    : {(jarak < 0).mean():.1%}   | Over-prioritize: {(jarak > 0).mean():.1%}')
print(f'Quadratic weighted kappa: {cohen_kappa_score(y.map(IDX), pd.Series(pred_cv).map(IDX), weights="quadratic"):.3f}')

### 7.4 Batas atas & learning curve
- **Batas atas:** seberapa sering aturan asli (kunci jawaban) cocok dengan label? Karena ada *noise* label, model mana pun tidak mungkin melewati angka ini.
- **Learning curve:** apakah model akan membaik kalau datanya ditambah?

In [ ]:
skor_asli = df.apply(true_priority_score, axis=1)
print(f'Batas atas (aturan asli vs label): {(skor_asli.apply(score_to_label) == y).mean():.3f}')

In [ ]:
plt.figure(figsize=(7, 3.6))
for nama in ['Random Forest', 'Logistic Regression']:
    nl, _, su = learning_curve(kandidat[nama], X, y, train_sizes=[0.2, 0.4, 0.6, 0.8, 1.0],
                               cv=RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=0))
    plt.plot(nl, su.mean(axis=1), 'o-', label=nama)
plt.xlabel('jumlah data latih'); plt.ylabel('akurasi CV'); plt.title('Learning curve'); plt.legend(); plt.show()

**Kesimpulan evaluasi**
| Pertanyaan | Jawaban |
|---|---|
| Lebih baik dari baseline? | Ya: Dummy ±0,38 < aturan ±0,59 < satu pohon ±0,66 < **RF ±0,73** |
| Seberapa parah kesalahannya? | ±99% meleset paling jauh 1 tingkat; kappa berbobot ±0,86 |
| Risiko utama? | *Under-prioritize* ±19%, hampir dua kali *over-prioritize*. Diperbaiki di Bagian 9 |
| Model terbaik? | **Logistic Regression ±0,85**, menang di hampir semua split, dan unggul di semua ukuran data (learning curve). Aturan pembuat label berbentuk **penjumlahan berbobot**, struktur yang cocok untuk model linear; pohon harus meniru garis miring dengan "anak tangga" dan butuh jauh lebih banyak data |
| Ruang perbaikan? | Aturan asli mencapai ±0,98, jadi kesenjangan 25 poin berasal dari keterbatasan model/data, bukan noise |

Random Forest tetap menjadi model utama karena **diwajibkan brief**; keunggulan Logistic Regression dicatat sebagai rekomendasi (dan sebagian mencerminkan cara data sintetis dibuat).

## 8. Penjelasan Model

### 8.1 Feature importance: dua cara mengukur, dua jawaban

In [ ]:
hutan, pre = rf.named_steps['model'], rf.named_steps['pre']
imp = pd.Series(hutan.feature_importances_, index=pre.get_feature_names_out()).sort_values()
perm = permutation_importance(rf, X_test, y_test, n_repeats=50, random_state=0)
pi = pd.Series(perm.importances_mean, index=X_test.columns).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
imp.plot.barh(ax=axes[0], color='#4C72B0'); axes[0].set_title('Impurity importance (data latih)')
pi.plot.barh(ax=axes[1], color='#DD8452', xerr=perm.importances_std[np.argsort(perm.importances_mean)])
axes[1].set_title('Permutation importance (data uji)'); plt.tight_layout(); plt.show()

`estimated_hours` tampak penting menurut impurity, tetapi kecil menurut permutation. Impurity importance **bias** ke fitur bernilai banyak, dan informasi jam kerja juga tumpang-tindih dengan `complexity` dan `task_category`. Pelajarannya: "penting" bergantung pada **cara mengukurnya**.

### 8.2 SHAP
SHAP berasal dari *Shapley value* (teori permainan): selisih "prediksi tugas ini − prediksi rata-rata" dibagi secara adil ke tiap fitur, sehingga $\text{prediksi} = \text{dasar} + \sum \phi_{\text{fitur}}$ (additivity).

In [ ]:
X_enc = pre.transform(X)
explainer = shap.TreeExplainer(hutan)
sv = explainer.shap_values(X_enc)
sv = np.stack(sv, axis=-1) if isinstance(sv, list) else np.asarray(sv)     # (tugas, fitur, kelas)
dasar = np.asarray(explainer.expected_value)
kelas = list(hutan.classes_)
print('Cek additivity, selisih maks:', np.abs(dasar + sv.sum(axis=1) - hutan.predict_proba(X_enc)).max())

ci = kelas.index('Critical')
shap.summary_plot(sv[:, :, ci], X_enc, show=False, plot_size=(8, 4.2))
plt.title('SHAP untuk kelas Critical'); plt.show()

In [ ]:
proba_crit = hutan.predict_proba(X_enc)[:, ci]
j = max([k for k in X_test.index if y[k] == 'Critical'], key=lambda k: proba_crit[k])
print('Contoh tugas uji:', df.loc[j, ['task_name'] + FITUR].to_dict())
shap.plots.waterfall(shap.Explanation(values=sv[j, :, ci], base_values=dasar[ci], data=X_enc.iloc[j].values,
                                      feature_names=list(X_enc.columns)), show=False)
plt.title('Kenapa tugas ini diprediksi Critical?'); plt.show()

### 8.3 Validasi: apakah model menemukan aturan rahasia? 🕵️
"Pengaruh sebenarnya" tiap faktor = **sebaran** suku rumusnya (faktor yang hampir konstan tidak bisa membedakan prioritas). Dari sisi model, dipakai SHAP untuk **tingkat prioritas harapan** $\mathbb{E}[\text{tingkat}] = \sum_k k \cdot P(k)$, dengan fitur dikelompokkan sesuai suku rumus.

In [ ]:
pr = (df['estimated_hours'] / (df['deadline_days'] * HOURS_PER_DAY)).clip(upper=1.5) / 1.5
kebenaran = pd.DataFrame({'pressure': 0.40 * pr, 'impact': 0.30 * (df['impact_score'] - 1) / 4,
                          'urgency': 0.15 * (df['urgency'] - 1) / 4,
                          'dependency': 0.15 * df['dependency_count'].clip(upper=4) / 4}).std()
kebenaran['tidak ada di aturan'] = 0.0

TINGKAT = np.array([ORDER.index(c) for c in kelas])
sv_E = (sv * TINGKAT).sum(axis=2)
kol = list(X_enc.columns)
grup = {'pressure': ['deadline_days', 'estimated_hours'], 'impact': ['impact_score'], 'urgency': ['urgency'],
        'dependency': ['dependency_count'],
        'tidak ada di aturan': ['complexity'] + [c for c in kol if c.startswith('task_category')]}
model_shap = pd.Series({k: np.abs(sv_E[:, [kol.index(c) for c in v]].sum(axis=1)).mean() for k, v in grup.items()})

banding = pd.DataFrame({'aturan asli': kebenaran / kebenaran.sum(), 'model (SHAP)': model_shap / model_shap.sum()})
banding.plot.barh(figsize=(7, 3.6), color=['#55A868', '#4C72B0']); plt.gca().invert_yaxis()
plt.title('Porsi pengaruh: aturan asli vs penjelasan model'); plt.show()
(banding * 100).round(0)

- ✅ **pressure** dan **impact** = dua faktor terbesar, sesuai aturan asli. **dependency** paling kecil, juga sesuai.
- ⚠️ **urgency** sedikit dilebih-lebihkan: ia ikut mengambil kredit deadline (korelasi −0,90).
- ⚠️ ±12% pengaruh jatuh ke fitur yang **tidak ada di aturan** (`task_category`, `complexity`). Model memakainya sebagai *proxy* karena berkorelasi dengan faktor asli.

### 8.4 Audit kualitas label
Tugas yang labelnya diberi peluang sangat kecil oleh model (lewat cross-validation) adalah **tersangka salah label**. Dicek terhadap label yang sengaja digeser generator. ⏱️ ±20 detik.

In [ ]:
pre_lr = ColumnTransformer([('num', StandardScaler(), FITUR_ANGKA),
                            ('ord', OrdinalEncoder(categories=[['Low', 'Medium', 'High']]), FITUR_ORDINAL),
                            ('nom', OneHotEncoder(), FITUR_NOMINAL)])
P = sum(cross_val_predict(Pipeline([('pre', pre_lr), ('m', LogisticRegression(max_iter=3000, C=10))]), X, y,
                          cv=StratifiedKFold(5, shuffle=True, random_state=r), method='predict_proba') for r in range(5)) / 5
p_label = pd.Series(P[np.arange(len(y)), [sorted(ORDER).index(v) for v in y]], index=df['task_id'])

aturan_label = truth_df.set_index('task_id').apply(lambda r: score_to_label(true_priority_score(r)), axis=1)
digeser = set(aturan_label.index[aturan_label != truth_df.set_index('task_id')['priority_level']])
top10 = p_label.sort_values().head(10)
print('Label yang sengaja digeser generator:', sorted(digeser))
print('Tertangkap di 10 tersangka teratas  :', len(set(top10.index) & digeser), 'dari', len(digeser))

Tersangka bukan vonis. Label **tidak** diubah otomatis; dalam praktik, 10 tugas ini diserahkan ke manusia untuk dicek ulang.

## 9. Dari Prediksi ke Keputusan

### 9.1 Model final
Setelah resep teruji, model **dilatih ulang pada seluruh 200 tugas**. Perkiraan performanya diambil dari cross-validation di atas.

### 9.2 Lapisan keputusan berbasis biaya
Kesalahan tidak sama mahalnya: menganggap tugas darurat sebagai biasa jauh lebih buruk daripada sebaliknya.
$$C(i,j) = |i-j| \times \begin{cases}2 & j<i \ (\text{under})\\ 1 & j>i \ (\text{over})\end{cases} \qquad \hat{j} = \arg\min_j \sum_i P(i\mid\text{tugas})\,C(i,j)$$
Model memilih tingkat dengan **biaya harapan terkecil**, bukan sekadar peluang tertinggi.

### 9.3 Skor prioritas 0–100
Untuk mengurutkan to-do list: $\text{skor} = \tfrac{100}{3}\sum_k k\cdot P(k)$, dengan 0 = pasti Low dan 100 = pasti Critical.

In [ ]:
model_final = buat_pipeline(RandomForestClassifier(n_estimators=100, random_state=42)).fit(X, y)
hutan_f, pre_f = model_final.named_steps['model'], model_final.named_steps['pre']
explainer_f = shap.TreeExplainer(hutan_f)
TINGKAT_F = np.array([ORDER.index(c) for c in hutan_f.classes_])
K = len(ORDER)
BIAYA = np.array([[abs(i - j) * (2 if j < i else 1) for j in range(K)] for i in range(K)], dtype=float)

def proba_urut(model, data):
    return model.predict_proba(data)[:, [list(model.classes_).index(k) for k in ORDER]]

def putuskan(P, biaya=BIAYA):
    return (P @ biaya).argmin(axis=1)

# Uji lapisan biaya dengan prediksi cross-validation (5 fold × 5 ulangan)
P_cv = sum(cross_val_predict(buat_pipeline(RandomForestClassifier(n_estimators=100, random_state=42)), X, y,
                             cv=StratifiedKFold(5, shuffle=True, random_state=r),
                             method='predict_proba')[:, [sorted(ORDER).index(k) for k in ORDER]] for r in range(5)) / 5
benar = y.map(IDX).values
def ringkas(p):
    return {'akurasi': (p == benar).mean(), 'recall Critical': ((p == 3) & (benar == 3)).sum() / (benar == 3).sum(),
            'under-prioritize': (p < benar).mean(), 'over-prioritize': (p > benar).mean(),
            'biaya rata-rata': BIAYA[benar, p].mean()}
print(pd.DataFrame({'argmax': ringkas(P_cv.argmax(axis=1)), 'biaya minimum': ringkas(putuskan(P_cv))}).round(3))

skor_model = 100 / 3 * (P_cv @ np.arange(K))
print(f'\nSpearman(skor model, skor asli) = {spearmanr(skor_model, skor_asli).statistic:.3f}')
print(f'Spearman(label,      skor asli) = {spearmanr(benar, skor_asli).statistic:.3f}')

- Lapisan biaya: *under-prioritize* turun **±19% → ±9%**, recall Critical naik, akurasi tidak turun, dan biaya rata-rata turun ±27%. Harganya, *over-prioritize* naik; itu tukar-menukar yang disengaja.
- Skor 0–100 mengurutkan tugas **sedikit lebih dekat** ke kebenaran tersembunyi daripada label 4 tingkat itu sendiri. Untuk mengurutkan, skor kontinu lebih baik daripada label.

In [ ]:
def _kalimat(f, v):
    return {'deadline_days': lambda v: f'deadline {int(v)} hari lagi', 'urgency': lambda v: f'rasa mendesak {int(v)}/5',
            'impact_score': lambda v: f'dampak {int(v)}/5', 'estimated_hours': lambda v: f'butuh ±{float(v):g} jam',
            'dependency_count': lambda v: f'{int(v)} tugas bergantung padanya', 'complexity': lambda v: f'kesulitan {v}',
            'task_category': lambda v: f'kategori {v}'}[f](v)

def prioritaskan(tugas, n_alasan=3):
    """Terima tabel tugas (kolom FITUR, opsional 'nama'); kembalikan keputusan, skor, keyakinan, alasan."""
    data = tugas[FITUR]
    P = proba_urut(model_final, data)
    kep, skor = putuskan(P), 100 / 3 * (P @ np.arange(K))
    Xe = pre_f.transform(data)
    s = explainer_f.shap_values(Xe)
    s = np.stack(s, axis=-1) if isinstance(s, list) else np.asarray(s)
    s_skor = pd.DataFrame((s * TINGKAT_F).sum(axis=2) * 100 / 3, columns=Xe.columns)
    s_mentah = pd.DataFrame({f: s_skor[[c for c in Xe.columns if c == f or c.startswith(f + '_')]].sum(axis=1) for f in FITUR})
    baris = []
    for r in range(len(data)):
        top = s_mentah.iloc[r].sort_values(key=abs, ascending=False).head(n_alasan)
        baris.append({'nama': tugas['nama'].iloc[r] if 'nama' in tugas else f'tugas {r+1}',
                      'keputusan': ORDER[kep[r]], 'skor': round(float(skor[r]), 1),
                      'keyakinan': f'{P[r].max():.0%}' + ('  ⚠️ borderline' if P[r].max() < 0.6 else ''),
                      **{f'P({k})': round(float(P[r, i]), 2) for i, k in enumerate(ORDER)},
                      'alasan utama': ' | '.join(f"{'▲' if v > 0 else '▼'} {_kalimat(f, data.iloc[r][f])} ({v:+.0f})"
                                                  for f, v in top.items())})
    return pd.DataFrame(baris).sort_values('skor', ascending=False).reset_index(drop=True)

### 9.4 Hard vs soft: `.predict` vs `.predict_proba` untuk satu tugas baru
- `.predict` (*hard*) → **satu jawaban**: kelas dengan peluang tertinggi.
- `.predict_proba` (*soft*) → **peluang tiap kelas**: memperlihatkan seberapa yakin model.
- Lapisan biaya (9.2) memakai versi *soft* untuk mengambil keputusan.

In [ ]:
satu = pd.DataFrame([{'task_category': 'Tugas Kuliah', 'deadline_days': 4, 'urgency': 3, 'impact_score': 4,
                      'estimated_hours': 6.0, 'complexity': 'High', 'dependency_count': 1}])
P_satu = proba_urut(model_final, satu)
print('Hard  (.predict)       :', model_final.predict(satu)[0])
print('Soft  (.predict_proba) :', {k: round(float(p), 2) for k, p in zip(ORDER, P_satu[0])})
print('Keputusan biaya minimum:', ORDER[putuskan(P_satu)[0]])
print(f'Skor prioritas         : {100 / 3 * (P_satu @ np.arange(K))[0]:.1f}/100')

`.predict` hanya menjawab **High**. `.predict_proba` memperlihatkan bahwa peluang **Critical** juga cukup besar (±0,30). Informasi ini hilang kalau hanya memakai `.predict`, padahal justru inilah yang membuat skor 0–100 bisa membedakan tugas ini dari tugas High yang "aman".

## 10. Aplikasi Interaktif & Demo 3 Skenario

### 10.1 Form input (ipywidgets)
- **Skenario cepat:** satu klik mengisi form dengan salah satu dari 3 skenario brief, lalu langsung memprediksi.
- **🔮 Prediksi:** menampilkan keputusan, skor, peluang tiap kelas, dan alasan (dari SHAP).
- **➕ Tambah ke daftar:** memasukkan tugas ke to-do list yang otomatis terurut.

> ⚠️ Widget hanya tampil saat notebook **dijalankan** di Colab. Kalau notebook dibuka tanpa dijalankan, form tidak terlihat. Bukti demo yang tetap tersimpan ada di 10.2.
>
> 📸 **Untuk showcase:** klik ketiga tombol skenario satu per satu, lalu ambil screenshot atau rekam layar.

In [ ]:
import ipywidgets as w
from IPython.display import display, clear_output

SKENARIO = {
    '😌 Santai': {'nama': 'Cuci baju & beres kamar', 'task_category': 'Pribadi', 'deadline_days': 14, 'urgency': 1,
                 'impact_score': 1, 'estimated_hours': 1.5, 'complexity': 'Low', 'dependency_count': 0},
    '🤔 Borderline': {'nama': 'Laporan praktikum', 'task_category': 'Praktikum', 'deadline_days': 5, 'urgency': 3,
                     'impact_score': 3, 'estimated_hours': 4.0, 'complexity': 'Medium', 'dependency_count': 1},
    '🚨 H-1 UAS': {'nama': 'Belajar UAS besok', 'task_category': 'Kuis/Ujian', 'deadline_days': 1, 'urgency': 5,
                  'impact_score': 5, 'estimated_hours': 8.0, 'complexity': 'High', 'dependency_count': 0},
}

gaya, lebar = {'description_width': '140px'}, w.Layout(width='420px')
inp = {
    'nama'            : w.Text(value='Tugas baru', description='Nama tugas', style=gaya, layout=lebar),
    'task_category'   : w.Dropdown(options=['Tugas Kuliah', 'Praktikum', 'Kuis/Ujian', 'Organisasi', 'Pribadi'],
                                   description='Kategori', style=gaya, layout=lebar),
    'deadline_days'   : w.IntSlider(value=7, min=1, max=30, description='Deadline (hari)', style=gaya, layout=lebar),
    'urgency'         : w.IntSlider(value=3, min=1, max=5, description='Rasa mendesak', style=gaya, layout=lebar),
    'impact_score'    : w.IntSlider(value=3, min=1, max=5, description='Dampak', style=gaya, layout=lebar),
    'estimated_hours' : w.FloatSlider(value=3.0, min=0.5, max=20, step=0.5, description='Perkiraan jam', style=gaya, layout=lebar),
    'complexity'      : w.Dropdown(options=['Low', 'Medium', 'High'], value='Medium', description='Kesulitan', style=gaya, layout=lebar),
    'dependency_count': w.IntSlider(value=0, min=0, max=5, description='Tugas yang menunggu', style=gaya, layout=lebar),
}
b_pred, b_add, b_reset = (w.Button(description='🔮 Prediksi', button_style='primary'),
                          w.Button(description='➕ Tambah ke daftar', button_style='success'),
                          w.Button(description='🗑️ Kosongkan daftar'))
b_preset = [w.Button(description=nama, layout=w.Layout(width='135px')) for nama in SKENARIO]
out_hasil, out_daftar, daftar = w.Output(), w.Output(), []

baca = lambda: pd.DataFrame([{k: v.value for k, v in inp.items()}])

def isi_form(nilai):
    for k, v in nilai.items():
        inp[k].value = v

def saat_prediksi(_=None):
    with out_hasil:
        clear_output(wait=True)
        h = prioritaskan(baca()).iloc[0]
        print(f"Keputusan: {h['keputusan']}  |  Skor: {h['skor']}/100  |  Keyakinan: {h['keyakinan']}")
        print('Alasan   :', h['alasan utama'])
        fig, ax = plt.subplots(figsize=(5, 1.8))
        ax.barh(ORDER, [h[f'P({k})'] for k in ORDER], color=[WARNA[k] for k in ORDER])
        ax.set_xlim(0, 1); ax.invert_yaxis(); ax.set_title('Peluang tiap kelas'); plt.show()

def tampilkan():
    with out_daftar:
        clear_output(wait=True)
        if not daftar: print('Daftar masih kosong.'); return
        u = prioritaskan(pd.concat(daftar, ignore_index=True)); u.index += 1
        print('📋 TO-DO LIST (urut skor prioritas):'); display(u[['nama', 'keputusan', 'skor', 'keyakinan']])

for b in b_preset:
    b.on_click(lambda tombol: (isi_form(SKENARIO[tombol.description]), saat_prediksi()))
b_pred.on_click(saat_prediksi)
b_add.on_click(lambda _: (daftar.append(baca()), tampilkan()))
b_reset.on_click(lambda _: (daftar.clear(), tampilkan()))
display(w.VBox([w.HTML('<h3>🗂️ AI Task Prioritizer</h3>'),
                w.HBox([w.Label('Skenario cepat:'), *b_preset]), *inp.values(),
                w.HBox([b_pred, b_add, b_reset]), out_hasil, out_daftar]))

### 10.2 Demo 3 skenario lewat form
Sel ini menjalankan **jalur yang sama persis** dengan menekan tombol di form: `isi_form()` mengisi widget, lalu `baca()` membaca nilai dari widget dan meneruskannya ke `prioritaskan()`. Bedanya, hasilnya dicetak sebagai output biasa, sehingga **tetap tersimpan** dan bisa dibaca penilai tanpa menjalankan notebook.

In [ ]:
for nama, nilai in SKENARIO.items():
    isi_form(nilai)                      # = menekan tombol skenario
    h = prioritaskan(baca()).iloc[0]     # = menekan 🔮 Prediksi (membaca nilai dari widget)
    print(f"{nama:14} → {h['keputusan']:8} | skor {h['skor']:5.1f}/100 | keyakinan {h['keyakinan']}")
    print(f"{'':16} P: " + ', '.join(f"{k} {h[f'P({k})']:.2f}" for k in ORDER))
    print(f"{'':16} alasan: {h['alasan utama']}\n")

**Bacaan 3 skenario**
| Skenario | Hasil | Artinya |
|---|---|---|
| 😌 Santai | **Low**, yakin ±100% | Dampak kecil, deadline jauh, kategori Pribadi: semua mendorong ke bawah |
| 🤔 Borderline | **High**, keyakinan ±68% (Medium ±27%) | Keyakinan terendah dari ketiganya: model melihat tugas ini di dekat batas Medium/High |
| 🚨 H-1 UAS | **Critical**, yakin ±98% | Deadline besok, rasa mendesak 5, dampak 5 |

Alasannya masuk akal secara manusiawi, dan arahnya (▲/▼) sesuai intuisi. Itu tanda model tidak sekadar menebak.

### 10.3 Mengurutkan satu to-do list penuh

In [ ]:
skenario = pd.DataFrame(list(SKENARIO.values()))
todo = pd.concat([skenario, pd.DataFrame([
    {'nama': 'Tugas kelompok Basdat', 'task_category': 'Tugas Kuliah', 'deadline_days': 3, 'urgency': 4,
     'impact_score': 4, 'estimated_hours': 6.0, 'complexity': 'High', 'dependency_count': 3},
    {'nama': 'Rapat divisi', 'task_category': 'Organisasi', 'deadline_days': 2, 'urgency': 4,
     'impact_score': 2, 'estimated_hours': 1.5, 'complexity': 'Low', 'dependency_count': 2},
    {'nama': 'Kuis Statprob', 'task_category': 'Kuis/Ujian', 'deadline_days': 6, 'urgency': 3,
     'impact_score': 3, 'estimated_hours': 3.0, 'complexity': 'Medium', 'dependency_count': 0},
])], ignore_index=True)
pd.set_option('display.max_colwidth', None)
urutan = prioritaskan(todo); urutan.index += 1
urutan[['nama', 'keputusan', 'skor', 'keyakinan', 'alasan utama']]

### 10.4 Analisis batas: "bagaimana kalau ditunda?"
Tugas yang sama disimulasikan dengan sisa hari yang makin sedikit, untuk melihat **di titik mana** ia melewati batas menjadi Critical. Asumsinya, rasa mendesak ikut naik sesuai `urgency_harapan`.

In [ ]:
paper = {'task_category': 'Tugas Kuliah', 'impact_score': 4, 'estimated_hours': 8.0, 'complexity': 'High', 'dependency_count': 1}
sim = pd.DataFrame([dict(paper, deadline_days=h, urgency=urgency_harapan(h)) for h in range(14, 0, -1)])
P_sim = proba_urut(model_final, sim[FITUR])
sim['skor'], sim['keputusan'] = 100 / 3 * (P_sim @ np.arange(K)), [ORDER[j] for j in putuskan(P_sim)]

plt.figure(figsize=(8, 3.6))
plt.plot(sim['deadline_days'], sim['skor'], 'k-', lw=1)
plt.scatter(sim['deadline_days'], sim['skor'], c=[WARNA[k] for k in sim['keputusan']], s=60, zorder=3)
for k in ORDER: plt.scatter([], [], color=WARNA[k], label=k)
plt.gca().invert_xaxis(); plt.legend(fontsize=8)
plt.xlabel('sisa hari sampai deadline (waktu berjalan →)'); plt.ylabel('skor prioritas')
plt.title('Paper (impact 4, 8 jam): kapan menjadi darurat?'); plt.show()
print('Menjadi Critical saat sisa', sim.loc[sim['keputusan'] == 'Critical', 'deadline_days'].max(), 'hari')

Lompatan terbesar terjadi saat `urgency` naik sesuai asumsi. Ini menegaskan bahwa model banyak bersandar pada urgency, dan hasil simulasi bergantung pada asumsi tersebut.

## 11. Model Card & Kesimpulan

| Bagian | Isi |
|---|---|
| **Tujuan** | Membantu mahasiswa mengurutkan tugas berdasarkan prioritas dan skor 0–100, beserta alasannya |
| **Bukan untuk** | Keputusan akademik resmi atau menggantikan penilaian pribadi; ini **alat bantu** |
| **Data** | 200 tugas sintetis (seed 42); 10 sel diimputasi |
| **Model** | Imputasi median + ordinal/one-hot encoding + Random Forest (100 pohon), dilatih ulang pada 200 tugas |
| **Keputusan** | Biaya minimum; *under-prioritize* dihukum 2× lebih berat |
| **Performa (CV 5×10)** | Akurasi ±0,73; kappa berbobot ±0,86; ±99% kesalahan hanya 1 tingkat |
| **Pembanding** | Dummy ±0,38 · aturan ±0,59 · satu pohon ±0,66 · Logistic Regression ±0,85 |
| **Validasi eksternal** | SHAP menemukan faktor utama aturan asli; audit label menangkap 4/4 label yang digeser |
| **Uji tambahan** | Pilihan imputasi, kebocoran data, kemiripan latih–uji, jumlah pohon, dan Shapley manual (Lampiran A–F) |

**Keterbatasan & risiko**
1. **Data sintetis.** Polanya dibuat sendiri, belum tentu mencerminkan mahasiswa sungguhan.
2. **Data kecil.** 200 tugas, 27 Critical; performa bergoyang ±6 poin antar split.
3. **Random Forest bukan model terbaik** untuk data ini (Logistic Regression ±11 poin lebih akurat).
4. **Jalan pintas** ±12% lewat `task_category`/`complexity`.
5. **`urgency` diisi manusia.** Bisa menjadi bocoran jawaban kalau diisi setelah tahu prioritasnya.
6. **Kebocoran kecil:** statistik imputasi dihitung dari seluruh data (10 sel).
7. **Model tidak tahu progres pengerjaan.** `estimated_hours` sebaiknya diisi dengan **sisa** jam.

**Langkah berikutnya:** mengumpulkan data tugas nyata selama beberapa minggu, membandingkan ulang Random Forest dengan Logistic Regression, dan menguji matriks biaya bersama pengguna.

---
**Kesimpulan.** Proyek ini membangun alur ML lengkap, dari data mentah yang kotor sampai alat bantu keputusan yang bisa menjelaskan dirinya sendiri. Nilai tambah utamanya: setiap klaim (kualitas pembersihan, kejujuran evaluasi, kebenaran penjelasan) **dibuktikan dengan data**, bukan sekadar diasumsikan.

---
# 📎 Lampiran: Bukti untuk Setiap Klaim

Lampiran ini berisi kode yang membuktikan angka-angka yang disebut di alur utama. Semuanya ikut berjalan saat `Run all`.

| Lampiran | Isi | Mendukung bagian |
|---|---|---|
| A | Simulasi imputasi 200× + uji sensitivitas | 3.2, 3.3 |
| B | Mekanisme data hilang (MCAR / MAR / MNAR) | 3.2 |
| C | Mutual information + *bootstrap* selang kepercayaan | 4.4 |
| D | Kebocoran data, efek `stratify`, *adversarial validation*, `GroupKFold`, template vs Pipeline | 5.1, 5.2 |
| E | `predict` vs `predict_proba` pada data uji + skor OOB | 6, 9.4 |
| F | Shapley value dihitung manual | 8.2 |

## Lampiran A: Membuktikan pilihan imputasi
**Ide:** ambil baris yang jamnya **terisi** (jawabannya diketahui), sembunyikan 10% secara acak, isi dengan tiap metode, lalu hitung MAE (rata-rata selisih absolut dengan nilai asli). Ulangi 200 kali. ⏱️ ±30 detik.

In [ ]:
from sklearn.experimental import enable_iterative_imputer   # wajib sebelum import IterativeImputer
from sklearn.impute import KNNImputer, IterativeImputer

pra = df.set_index('task_id').mask(kosong).reset_index()     # data bersih, tapi sel yang tadinya kosong dikosongkan lagi
lengkap = pra.dropna(subset=['estimated_hours']).copy()
lengkap['urgency'] = lengkap['urgency'].fillna(lengkap['urgency'].median())
matriks = pd.concat([lengkap[FITUR_ANGKA], pd.get_dummies(lengkap['task_category']).astype(float)], axis=1)
rata, sd = matriks.mean(), matriks.std()
kj = FITUR_ANGKA.index('estimated_hours')

rng_a = np.random.default_rng(0)
hasil_mae = {m: [] for m in ['median global', 'median kategori', 'median kategori+impact', 'KNN (k=5)', 'Iterative']}
for _ in range(200):
    sembunyi = rng_a.random(len(lengkap)) < 0.10
    asli = lengkap.loc[sembunyi, 'estimated_hours'].values
    t_a = lengkap.copy(); t_a.loc[sembunyi, 'estimated_hours'] = np.nan
    jam = t_a['estimated_hours']
    tebak = {
        'median global'          : jam.fillna(jam.median()),
        'median kategori'        : jam.fillna(t_a.groupby('task_category')['estimated_hours'].transform('median')),
        'median kategori+impact' : jam.fillna(t_a.groupby(['task_category', 'impact_score'])['estimated_hours'].transform('median')),
    }
    m_a = matriks.copy(); m_a.loc[sembunyi, 'estimated_hours'] = np.nan
    knn = KNNImputer(n_neighbors=5).fit_transform((m_a - rata) / sd)[:, kj] * sd['estimated_hours'] + rata['estimated_hours']
    tebak['KNN (k=5)'] = pd.Series(knn, index=t_a.index)
    tebak['Iterative'] = pd.Series(IterativeImputer(random_state=0).fit_transform(m_a)[:, kj], index=t_a.index)
    for nama, isi in tebak.items():
        hasil_mae[nama].append(np.abs(isi.fillna(jam.median())[sembunyi].values - asli).mean())

mae = pd.DataFrame(hasil_mae)
mae.agg(['mean', 'std']).T.round(3).sort_values('mean')

**Uji sensitivitas.** Imputasi yang lebih akurat belum tentu membuat model lebih baik. Random Forest yang sama dilatih pada tiga versi data, dengan cross-validation 5×10. ⏱️ ±40 detik.

In [ ]:
def lengkapi_sisanya(t):
    t = t.copy()
    t['urgency'] = t['urgency'].fillna(t['urgency'].median()).round()
    t['complexity'] = t['complexity'].fillna(t['complexity'].mode()[0])
    return t

versi = {}
t_a = pra.copy(); t_a['estimated_hours'] = t_a['estimated_hours'].fillna(t_a['estimated_hours'].median())
versi['median global'] = lengkapi_sisanya(t_a)
t_a = pra.copy(); t_a['estimated_hours'] = t_a['estimated_hours'].fillna(
    t_a.groupby(['task_category', 'impact_score'])['estimated_hours'].transform('median'))
versi['median kategori+impact'] = lengkapi_sisanya(t_a)
versi['buang baris kosong'] = pra.dropna()

cv_a = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=0)
for nama, t_a in versi.items():
    skor = cross_val_score(buat_pipeline(RandomForestClassifier(n_estimators=100, random_state=42)),
                           t_a[FITUR], t_a['priority_level'], cv=cv_a)
    print(f'{nama:24} ({len(t_a)} baris)  akurasi CV = {skor.mean():.3f} ± {skor.std():.3f}')

**Bacaan:**
- **Median kategori + impact** (pilihan di 3.2) memberi MAE terkecil (±1,53 jam), setara KNN (±1,56) dan jauh lebih baik dari median global (±2,25).
- Tetapi untuk **model**, pilihan imputasi hampir tidak berpengaruh (0,728 vs 0,730), karena hanya 10 sel yang diisi. Membuang baris justru lebih buruk (0,708): 9 tugas hilang, termasuk informasi di kolom lain yang lengkap.

## Lampiran B: Kenapa datanya hilang?
| Jenis | Arti | Boleh diisi median? |
|---|---|---|
| **MCAR** | hilang benar-benar acak | ✅ aman |
| **MAR** | hilangnya bergantung pada **kolom lain** (misalnya kategori) | ⚠️ boleh, kalau diisi **per kelompok** |
| **MNAR** | hilangnya bergantung pada **nilai yang hilang itu sendiri** (tugas yang sangat lama justru tidak diisi) | ❌ median akan bias |

In [ ]:
hilang = raw['estimated_hours'].isna()
kategori_rapi = raw['task_category'].str.strip().str.title()
print('Jumlah jam kosong:', hilang.sum(), 'dari', len(raw))
pd.DataFrame({'jam kosong': kategori_rapi[hilang].value_counts(),
              'jam terisi': kategori_rapi[~hilang].value_counts()}).fillna(0).astype(int)

**Bacaan, dengan jujur:** 5 sel terlalu sedikit untuk **membuktikan** jenis hilangnya. Yang bisa dikatakan hanya: sel kosong tersebar di beberapa kategori, tidak menumpuk di satu tempat. Karena itu dipilih imputasi **per kelompok** (aman untuk MCAR maupun MAR). MNAR tidak bisa dideteksi dari data saja; satu-satunya jalan adalah menanyakan cara data dikumpulkan.

## Lampiran C: Mutual information & selang kepercayaan korelasi
**Mutual information** $I(X;Y) = H(Y) - H(Y\mid X)$: seberapa berkurang ketidakpastian tentang prioritas setelah tahu nilai fitur. Berbeda dengan Spearman, ia menangkap hubungan **tidak searah** dan kategori **tanpa urutan**.

In [ ]:
from sklearn.feature_selection import mutual_info_classif

fitur_mi = eda[NUM + ['pressure']].copy()
fitur_mi['complexity'] = eda['complexity'].map({'Low': 0, 'Medium': 1, 'High': 2})
fitur_mi['task_category'] = eda['task_category'].astype('category').cat.codes
diskrit = [c in ('urgency', 'impact_score', 'dependency_count', 'complexity', 'task_category') for c in fitur_mi.columns]
mi = pd.Series(mutual_info_classif(fitur_mi, eda['priority_level'], discrete_features=diskrit, random_state=0),
               index=fitur_mi.columns)
mi.sort_values(ascending=False).round(3)

**Bootstrap:** ambil ulang 200 tugas dengan pengembalian, hitung korelasinya, ulangi 1.000 kali. Rentang tengah 95% = selang kepercayaan.

In [ ]:
rng_c = np.random.default_rng(0)
kolom_ci = NUM + ['pressure']
boot = []
for _ in range(1000):
    sampel = eda.iloc[rng_c.integers(0, len(eda), len(eda))]
    boot.append(sampel[kolom_ci].corrwith(sampel['y'], method='spearman'))
ci_boot = pd.DataFrame(boot).quantile([0.025, 0.975]).T
ci_boot.columns = ['batas bawah', 'batas atas']
ci_boot.insert(0, 'korelasi', korelasi_target.reindex(ci_boot.index))
ci_boot.sort_values('korelasi').round(2)

**Bacaan:**
- `pressure` tetap paling informatif, konsisten dengan Spearman.
- `task_category` muncul cukup tinggi, di atas `estimated_hours`. Ini tidak bisa terlihat dengan Spearman, dan menjelaskan kenapa model memakai kategori sebagai jalan pintas (8.3).
- Selang `dependency_count` hampir menyentuh 0: hubungannya lemah dan belum meyakinkan. Ini konsisten dengan SHAP, yang memberinya pengaruh terkecil.

## Lampiran D: Jebakan dalam evaluasi
### D.1 Kebocoran: memilih fitur **sebelum** membagi data
Dibuat **1.000 fitur acak murni** dan **label yang diacak**, jadi tidak ada apa pun yang bisa dipelajari. Model yang jujur seharusnya hanya setara menebak asal.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

Z = pd.DataFrame(np.random.default_rng(1).normal(size=(200, 1000)))
y_acak = y.sample(frac=1, random_state=0).values
cv_d = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=0)

pilih = SelectKBest(f_classif, k=20).fit(Z, y_acak)                       # ❌ seleksi memakai SELURUH data
salah = cross_val_score(LogisticRegression(max_iter=2000), Z.loc[:, pilih.get_support()], y_acak, cv=cv_d)
benar_d = cross_val_score(Pipeline([('pilih', SelectKBest(f_classif, k=20)),  # ✅ seleksi di DALAM tiap fold
                                    ('model', LogisticRegression(max_iter=2000))]), Z, y_acak, cv=cv_d)
print(f'Tebak asal (kelas terbanyak): {pd.Series(y_acak).value_counts(normalize=True).max():.3f}')
print(f'❌ Seleksi sebelum CV        : {salah.mean():.3f}   <- tampak "belajar" dari noise')
print(f'✅ Seleksi di dalam CV       : {benar_d.mean():.3f}')

### D.2 Efek `stratify`: jumlah Critical di data uji untuk 1.000 acakan berbeda

In [ ]:
tanpa = [(train_test_split(X, y, test_size=0.2, random_state=s)[3] == 'Critical').sum() for s in range(1000)]
dengan = [(train_test_split(X, y, test_size=0.2, random_state=s, stratify=y)[3] == 'Critical').sum() for s in range(1000)]
print(f'Tanpa stratify : {min(tanpa)}–{max(tanpa)} tugas Critical')
print(f'Dengan stratify: {min(dengan)}–{max(dengan)} tugas Critical')

### D.3 *Adversarial validation*: apakah data uji mirip data latih?
Model dilatih untuk **membedakan** baris latih vs uji. AUC ≈ 0,5 berarti tidak bisa dibedakan (adil). Pembandingnya: AUC dari label latih/uji yang **diacak** 30 kali. ⏱️ ±40 detik.

In [ ]:
from sklearn.metrics import roc_auc_score

gabung = pd.concat([X_train, X_test])
asal = np.r_[np.zeros(len(X_train)), np.ones(len(X_test))]           # 0 = latih, 1 = uji
pembeda = buat_pipeline(RandomForestClassifier(n_estimators=100, random_state=0))

def auc_pembeda(label):
    p = cross_val_predict(pembeda, gabung, label, cv=StratifiedKFold(5, shuffle=True, random_state=0),
                          method='predict_proba')[:, 1]
    return roc_auc_score(label, p)

auc_asli = auc_pembeda(asal)
rng_d = np.random.default_rng(0)
auc_palsu = np.array([auc_pembeda(rng_d.permutation(asal)) for _ in range(30)])
print(f'AUC membedakan latih vs uji : {auc_asli:.3f}')
print(f'AUC dari label acak (30×)   : rata-rata {auc_palsu.mean():.3f}, batas 95% {np.quantile(auc_palsu, 0.95):.3f}')
print(f'Peluang AUC setinggi ini muncul karena kebetulan ≈ {(auc_palsu >= auc_asli).mean():.2f}')

### D.4 Kebocoran kelompok (`GroupKFold`)
Nama tugas yang sama (misalnya "Cuci baju") muncul di data latih **dan** uji. Kalau itu tugas yang sama, skor uji bisa terlalu bagus. Uji: bagi CV **per nama tugas**, sehingga nama yang sama tidak pernah terpisah.

In [ ]:
from sklearn.model_selection import GroupKFold

nama_latih, nama_uji = set(df.loc[X_train.index, 'task_name']), set(df.loc[X_test.index, 'task_name'])
print(f'{len(nama_uji & nama_latih)} dari {len(nama_uji)} nama tugas di data uji juga ada di data latih')
model_rf = buat_pipeline(RandomForestClassifier(n_estimators=100, random_state=42))
biasa = cross_val_score(model_rf, X, y, cv=RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=0))
grup_cv = cross_val_score(model_rf, X, y, cv=GroupKFold(n_splits=5), groups=df['task_name'])
print(f'CV biasa          : {biasa.mean():.3f}')
print(f'CV per nama tugas : {grup_cv.mean():.3f}')

### D.5 Template brief vs Pipeline (cross-validation 5×10)

In [ ]:
cv_t = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=0)
s_tpl = cross_val_score(RandomForestClassifier(n_estimators=100, random_state=42), X_tpl, y_tpl, cv=cv_t)
s_pip = cross_val_score(buat_pipeline(RandomForestClassifier(n_estimators=100, random_state=42)), X, y, cv=cv_t)
print(f'Template get_dummies: {s_tpl.mean():.3f} ± {s_tpl.std():.3f}')
print(f'Pipeline            : {s_pip.mean():.3f} ± {s_pip.std():.3f}')
print(f'Pipeline menang di {((s_pip - s_tpl) > 0).mean():.0%} split, selisih rata-rata {(s_pip - s_tpl).mean():+.3f}')

**Bacaan Lampiran D:**
- **D.1:** seleksi sebelum CV memberi ±0,49 dari data yang 100% acak, jauh di atas tebakan asal (±0,38). Seleksi yang benar turun ke ±0,26. Kebocoran bisa "menciptakan" akurasi dari nol.
- **D.2:** tanpa `stratify`, jumlah Critical di data uji bisa 0 sampai 12. Dengan `stratify`, selalu 5.
- **D.3:** AUC 0,58 masih di dalam rentang kebetulan (peluang ±13%), jadi tidak ada bukti data uji berbeda dari data latih.
- **D.4:** banyak nama tugas yang sama di latih dan uji, tetapi itu tugas **berbeda** (deadline, jam, dan dampaknya beda). CV per nama tugas hampir sama (±0,71 vs ±0,72), jadi tidak ada kebocoran kelompok.
- **D.5:** akurasi template dan Pipeline setara. Keunggulan Pipeline ada di keamanan pemakaian (5.1), bukan akurasi.

## Lampiran E: `predict` vs `predict_proba` & skor OOB
Baris dengan peluang tertinggi di bawah ±50% adalah tugas **borderline**: model sendiri ragu.

In [ ]:
proba_uji = pd.DataFrame(rf.predict_proba(X_test), columns=rf.classes_)[ORDER].round(2)
contoh = proba_uji.head(8).copy()
contoh['predict'] = rf.predict(X_test.head(8))
contoh['sebenarnya'] = y_test.head(8).values
contoh

**OOB (*out-of-bag*):** tiap pohon tidak melihat ±36,8% data latih, sehingga tiap tugas bisa dinilai oleh pohon-pohon yang tidak pernah melihatnya. Ini memberi perkiraan akurasi tanpa menyentuh data uji.

In [ ]:
jumlah_pohon = [10, 25, 50, 100, 200, 400]
oob = [buat_pipeline(RandomForestClassifier(n_estimators=k, oob_score=True, random_state=42))
       .fit(X_train, y_train).named_steps['model'].oob_score_ for k in jumlah_pohon]
plt.figure(figsize=(6, 3.2)); plt.plot(jumlah_pohon, oob, 'o-')
plt.xlabel('jumlah pohon'); plt.ylabel('akurasi OOB'); plt.title('Akurasi OOB vs jumlah pohon'); plt.show()
print(dict(zip(jumlah_pohon, np.round(oob, 3))))

**Bacaan:** akurasi OOB naik cepat dari 10 ke ±50–100 pohon, lalu naik sangat pelan (±0,69 → ±0,71 dari 100 ke 400 pohon). Menambah pohon tidak menyebabkan *overfitting*, hanya memperlambat. 100 pohon (sesuai brief) adalah kompromi yang wajar.

## Lampiran F: Shapley value dihitung manual
Untuk satu tugas uji, peluang Critical dijelaskan dengan rumus Shapley **persis**, memakai semua $2^7 = 128$ kombinasi fitur:
$$\phi_f = \sum_{S \subseteq F \setminus \{f\}} \frac{|S|!\,(p-|S|-1)!}{p!}\,\big[v(S \cup \{f\}) - v(S)\big]$$
- $v(S)$ = rata-rata peluang Critical jika hanya fitur di $S$ yang diketahui (fitur lain diambil dari 50 tugas latih acak)
- pecahan di depan = peluang tim $S$ persis yang sudah ada sebelum $f$ masuk, kalau urutan masuk diacak ($p$ = 7 fitur)

In [ ]:
from itertools import combinations
from math import factorial

x_f = X_test.iloc[[2]]
latar = X_train.sample(50, random_state=0)
k_crit = list(rf.classes_).index('Critical')
F_list, p_f = list(X.columns), len(X.columns)

def v(S):
    campuran = latar.copy()
    for f in S:
        campuran[f] = x_f[f].values[0]
    return rf.predict_proba(campuran)[:, k_crit].mean()

v_semua = {frozenset(S): v(S) for r in range(p_f + 1) for S in combinations(F_list, r)}
phi = {f: sum(factorial(len(S)) * factorial(p_f - len(S) - 1) / factorial(p_f) * (v_semua[S | {f}] - v_semua[S])
              for S in v_semua if f not in S) for f in F_list}

print(pd.Series(phi).sort_values(key=abs, ascending=False).round(3).to_string())
dasar_m = v_semua[frozenset()]
print(f'\ndasar {dasar_m:.3f} + jumlah kontribusi {sum(phi.values()):+.3f} = {dasar_m + sum(phi.values()):.3f}')
print(f'prediksi model peluang Critical              = {rf.predict_proba(x_f)[0, k_crit]:.3f}')

**Bacaan:** dasar ditambah semua kontribusi **persis sama** dengan prediksi model (*additivity*). Library `shap` (Bagian 8.2) melakukan hal yang sama dengan algoritma cepat khusus pohon, tanpa menghitung semua kombinasi.

---
## Versi Library
Dicatat supaya hasilnya bisa diulang. Kalau versi library berbeda, angka bisa sedikit bergeser.

In [ ]:
import platform, sklearn, scipy, matplotlib
print('Python'.ljust(14), platform.python_version())
for m in [np, pd, sklearn, scipy, matplotlib, sns, shap, w]:
    print(m.__name__.ljust(14), m.__version__)